# Pilotos v5 — RoBERTuito (IroSvA): congelamiento de capas y label smoothing

Notebook **independiente** de `RoBERTuito_GroupNestedCV_5x3_TPE20_v4.ipynb`. Copia sin cambios sus celdas de librerías, datos, grupos, modelo y trainer, y solo agrega la sección de pilotos. **No ejecuta la Nested CV, no reoptimiza y no toca el test.**

**Qué hace.** Reentrena la configuración final v4 (`robertuito_config_final.json`) en los mismos 3 Inner Folds y con las mismas semillas que el diagnóstico de convergencia (16-bis), sin early stopping y con 4 evaluaciones por época:

| Piloto | Cambio respecto a la v4 | Respaldo |
|---|---|---|
| P0 | Ninguno (línea base en esta misma máquina) | — |
| P1 | Congelar embeddings + capas 0–2 (3 de 12, ~25 %) | Ingle et al. (2022); Merchant et al. (2020) |
| P2 | Congelar embeddings + capas 0–5 (6 de 12, 50 %) | Ingle et al. (2022); Merchant et al. (2020) |
| P3 | Label smoothing = 0 (sin congelar) | Desai & Durrett (2020) |

Variantes: **MX y CU**. Total: 4 pilotos × 2 variantes × 3 Inner Folds = **24 fine-tunings**.

**Criterio fijado antes de ver resultados:** una opción entra al espacio de búsqueda de la v5 si **no baja el F1-Macro de validación máximo más de 0.01** frente a P0 (`no_daña_f1 = True`). Además se reporta la brecha de F1 train–validación, la subida de la pérdida de validación y el ECE.

**Cómo correrlo (Windows con GPU):**
1. Copia este notebook a la carpeta `notebooks/` del proyecto (junto al v4), para que encuentre `../data/`.
2. Reinicia el kernel y ejecuta **Run All**.
3. Si se interrumpe, vuelve a ejecutar: reutiliza los folds ya terminados.

**Salidas** (en `data/resultados_robertuito_grouped_5x3_tpe20_v4/`):
- `pilotos_v5/P*/<variante>_inner<k>.csv`: curvas por fold
- `robertuito_pilotos_convergencia.csv`: todas las curvas
- `robertuito_pilotos_resumen.csv`: tabla de decisión
- `robertuito_pilotos_curvas.png/.pdf`: figura de pérdida y F1

La corrida v4 no se modifica: el seguimiento de los pilotos va a `seguimiento/r5x3t20_v4/pilotos/`.


## 1. Librerías y entorno (igual que v4)

In [1]:
import os
import re

# Declarar antes de importar Transformers/Hugging Face para ocultar barras de descarga.
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")

import gc
import time
import json
import pickle
import html
import random
import tempfile
import shutil
import inspect
import warnings
from pathlib import Path
from datetime import datetime
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import optuna

from torch.utils.data import Dataset, Subset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)
from IPython.display import display, HTML

import matplotlib

matplotlib.use("Agg")  # Evita abrir una ventana por gráfica durante cientos de fits.
import matplotlib.pyplot as plt

import transformers
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
    set_seed,
)
from transformers.trainer_callback import PrinterCallback, ProgressCallback
from transformers.trainer_utils import get_last_checkpoint

try:
    from huggingface_hub.utils import disable_progress_bars

    disable_progress_bars()
except (ImportError, AttributeError):
    pass
transformers.logging.set_verbosity_error()
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)

VERSIONES = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "optuna": optuna.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}
print("Versiones:", " | ".join(f"{k} {v}" for k, v in VERSIONES.items()))
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

Versiones: torch 2.14.0+cu130 | transformers 5.17.0 | optuna 5.0.0 | numpy 1.26.4 | pandas 3.0.3
CUDA: True
GPU: NVIDIA RTX 6000 Ada Generation
VRAM (GB): 48.0


## 2. Configuración (igual que v4, salvo `RUN_DIR` y las banderas de ejecución)

In [2]:
MODEL_NAME = "pysentimiento/robertuito-base-uncased"

DATA_DIR = Path("../data/transformers")
RESULTS_DIR = Path("../data/resultados_robertuito_grouped_5x3_tpe20_v4")
PROGRESS_DIR = RESULTS_DIR / "seguimiento"

TEXT_COLS = {"PYSENTIMIENTO": "MESSAGE_TRANSFORMER"}
LABEL_COL = "IS_IRONIC"
ID_COL = "ID"
ORIGINAL_COL = "MESSAGE"
GROUP_COL = "GROUP_ID"

MAX_LENGTH = 128
SEED = 42

# ------------------------------------------------------------
# "prueba" valida rápidamente todo el pipeline.
# "completo" = experimento definitivo 5×3 / TPE20.
# ------------------------------------------------------------
MODO = "completo"  # primero "prueba"; luego "completo" y reiniciar kernel

if MODO == "prueba":
    VARIANTES = ["mx"]
    N_OUTER = 2
    N_INNER = 2
    N_TRIALS = 2
    N_STARTUP_TRIALS = 1
elif MODO == "completo":
    VARIANTES = ["mx", "es", "cu"]
    N_OUTER = 5
    N_INNER = 3
    N_TRIALS = 20
    N_STARTUP_TRIALS = 5
else:
    raise ValueError("MODO debe ser 'prueba' o 'completo'.")

# RUN_TAG corto + subcarpetas compactas para Windows.
# v4: carpeta nueva (entrenamiento revisado tras el diagnóstico de convergencia).
RUN_TAG = "r5x3t20_v4"
RUN_DIR = PROGRESS_DIR / RUN_TAG / "pilotos"  # PILOTOS v5: no toca la corrida v4
TMP_DIR = RUN_DIR / "tmp"
CHECKPOINT_DIR = RUN_DIR / "ckpt"
FIT_STATE_DIR = RUN_DIR / "state"
CURVE_DIR = RUN_DIR / "curves"
GRAPH_DIR = RUN_DIR / "graphs"
AUDIT_DIR = RUN_DIR / "audit"

for carpeta in [
    RESULTS_DIR,
    PROGRESS_DIR,
    RUN_DIR,
    TMP_DIR,
    CHECKPOINT_DIR,
    FIT_STATE_DIR,
    CURVE_DIR,
    GRAPH_DIR,
    AUDIT_DIR,
]:
    carpeta.mkdir(parents=True, exist_ok=True)

# Flujo recomendado (sección 18): opcionalmente una corrida corta con MODO="prueba";
# luego MODO="completo". Nested CV y optimización final usan solo train y corren seguidas;
# el test oficial requiere además CONFIRMACION_TEST.
EJECUTAR_NESTED = False  # pilotos: no se ejecuta la Nested CV
EJECUTAR_OPTIMIZACION_FINAL = False  # pilotos: no se reoptimiza
EJECUTAR_TEST_OFICIAL = False
CONFIRMACION_TEST = ""

SEMILLAS_FINALES = [42, 52, 62, 72, 82]
CLEAN_COMPLETED_CHECKPOINTS = True

# Seguimiento: panel de progreso común a todos los notebooks de modelos.
NOMBRE_EXPERIMENTO = "RoBERTuito"
HEARTBEAT_SEGUNDOS = 30
# Pitido de Windows al terminar o detenerse una etapa larga (False para desactivarlo).
AVISO_SONORO = True

# Optuna TPE: 5 trials iniciales aleatorios y 15 adaptativos en modo completo.
# Espacio común a BETO, DistilBETO y RoBERTuito.
SEARCH_SPACE = {
    "preprocessing": ["PYSENTIMIENTO"],
    "learning_rate": [1e-5, 2e-5],  # v3: 2e-5..5e-5 (memorizaba en ~100 pasos)
    "batch_size": [16],  # fijo: probado en los pilotos B y C
    "num_train_epochs": [4, 6],  # máximo; decide el early stopping
    "weight_decay": [0.01, 0.1],
    "loss_mode": ["standard", "balanced"],
    "classifier_dropout": [0.1, 0.2, 0.3],
    "label_smoothing": [0.1],  # fijo (piloto B)
    "lr_decay": [0.9],  # fijo (piloto B)
}

# Early stopping de los Inner Folds: épocas sin mejora de F1 antes de detener.
PACIENCIA_EARLY_STOPPING = 2

SEARCH_SPACE_REQUERIDO = {
    "preprocessing",
    "learning_rate",
    "batch_size",
    "num_train_epochs",
    "weight_decay",
    "loss_mode",
    "classifier_dropout",
    "label_smoothing",
    "lr_decay",
}
faltantes_search_space = SEARCH_SPACE_REQUERIDO - set(SEARCH_SPACE)
if faltantes_search_space:
    raise KeyError(f"SEARCH_SPACE incompleto. Faltan: {sorted(faltantes_search_space)}")

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

fits_teoricos = len(VARIANTES) * N_OUTER * (N_TRIALS * N_INNER + 1)

print(f"[CONFIG][RoBERTuito] modo={MODO} | variantes={VARIANTES}")
print(
    f"[CONFIG][RoBERTuito] Nested={N_OUTER}×{N_INNER} | "
    f"trials={N_TRIALS} | startup={N_STARTUP_TRIALS}"
)
print(
    f"[CONFIG][RoBERTuito] fine-tunings teóricos={fits_teoricos} | "
    f"nested_activo={EJECUTAR_NESTED}"
)
print(f"[CONFIG][RoBERTuito] RUN_DIR={RUN_DIR.resolve()}")
print("[CONFIG][RoBERTuito] SEARCH_SPACE OK:", sorted(SEARCH_SPACE))

# Comprobación preventiva de longitud de ruta en Windows (peor caso del espacio).
_ruta_ckpt_ejemplo = (
    CHECKPOINT_DIR / "mx" / "outer1" / "P_l2_b16_e6_w10_b_d3_s1_r9" / "inner3" / "checkpoint-99999"
).resolve()

print(
    f"[CONFIG][RoBERTuito] longitud ruta checkpoint ejemplo="
    f"{len(str(_ruta_ckpt_ejemplo))} caracteres"
)

if os.name == "nt" and len(str(_ruta_ckpt_ejemplo)) >= 230:
    raise RuntimeError(
        "La ruta prevista para checkpoints sigue siendo demasiado larga en Windows. "
        "Mueve el repositorio a una ruta más corta antes de entrenar."
    )

[CONFIG][RoBERTuito] modo=completo | variantes=['mx', 'es', 'cu']
[CONFIG][RoBERTuito] Nested=5×3 | trials=20 | startup=5
[CONFIG][RoBERTuito] fine-tunings teóricos=915 | nested_activo=False
[CONFIG][RoBERTuito] RUN_DIR=C:\Users\I21308\Documents\DeteccionDeSarcasmo\data\resultados_robertuito_grouped_5x3_tpe20_v4\seguimiento\r5x3t20_v4\pilotos
[CONFIG][RoBERTuito] SEARCH_SPACE OK: ['batch_size', 'classifier_dropout', 'label_smoothing', 'learning_rate', 'loss_mode', 'lr_decay', 'num_train_epochs', 'preprocessing', 'weight_decay']
[CONFIG][RoBERTuito] longitud ruta checkpoint ejemplo=190 caracteres


## 3. Carga del train y verificación del preprocesamiento (igual que v4)

In [3]:
MANIFEST_PATH = DATA_DIR / "manifest_preprocesamiento_transformers.csv"
PIPELINE_VERSION_ESPERADA = "transformers_comun_v2"


def cargar_manifest_preprocesamiento():
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(
            f"Falta {MANIFEST_PATH}. Ejecuta primero "
            "`01_preprocesamiento_dl_transformers.ipynb`."
        )

    manifest = pd.read_csv(MANIFEST_PATH)

    requeridas = {"pipeline_version", "variante", "train_final"}
    faltantes = requeridas - set(manifest.columns)
    if faltantes:
        raise ValueError(f"Manifest incompleto; faltan columnas: {sorted(faltantes)}")

    versiones = set(manifest["pipeline_version"].astype(str))
    if versiones != {PIPELINE_VERSION_ESPERADA}:
        raise ValueError(
            f"Versión de preprocessing inesperada: {versiones}. "
            f"Se esperaba {PIPELINE_VERSION_ESPERADA}."
        )

    return manifest


MANIFEST_PREPROCESAMIENTO = cargar_manifest_preprocesamiento()


def validar_archivo_train_con_manifest(variante):
    ruta = DATA_DIR / f"train_transformers_{variante}.csv"
    if not ruta.exists():
        raise FileNotFoundError(f"Falta {ruta}. Ejecuta primero el preprocessing Transformer.")

    fila = MANIFEST_PREPROCESAMIENTO[MANIFEST_PREPROCESAMIENTO["variante"].astype(str) == variante]
    if len(fila) != 1:
        raise ValueError(f"{variante}: se esperaba exactamente una fila en el manifest.")

    return ruta, fila.iloc[0]


def cargar_train(variante):
    ruta, manifest_row = validar_archivo_train_con_manifest(variante)
    df = pd.read_csv(ruta)

    requeridas = [ID_COL, ORIGINAL_COL, LABEL_COL, *TEXT_COLS.values()]
    faltantes = [c for c in requeridas if c not in df.columns]
    if faltantes:
        raise ValueError(f"{variante}: faltan columnas {faltantes}")

    df[ID_COL] = df[ID_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    for col in [ORIGINAL_COL, *TEXT_COLS.values()]:
        df[col] = df[col].fillna("").astype(str)

    if df[ID_COL].duplicated().any():
        raise ValueError(f"{variante}: IDs duplicados en el train FINAL.")

    if not set(df[LABEL_COL].unique()).issubset({0, 1}):
        raise ValueError(f"{variante}: etiquetas fuera de 0/1.")

    vacios = {
        col: int(df[col].eq("").sum())
        for col in [ORIGINAL_COL, *TEXT_COLS.values()]
        if df[col].eq("").any()
    }
    if vacios:
        raise ValueError(
            f"{variante}: entradas vacías en el train final {vacios}. "
            "Vuelve a ejecutar `01_preprocesamiento_dl_transformers.ipynb` "
            "(MESSAGE_TRANSFORMER requiere `pysentimiento` instalado)."
        )

    esperado = int(manifest_row["train_final"])
    if len(df) != esperado:
        raise ValueError(f"{variante}: train={len(df)} pero manifest={esperado}.")

    return df.reset_index(drop=True)


for variante in ["mx", "es", "cu"]:
    tmp = cargar_train(variante)
    print(
        f"[DATA][RoBERTuito][{variante.upper()}] "
        f"n={len(tmp)} | "
        f"clases={tmp[LABEL_COL].value_counts().sort_index().to_dict()} | "
        "manifest=OK"
    )

[DATA][RoBERTuito][MX] n=2399 | clases={0: 1599, 1: 800} | manifest=OK
[DATA][RoBERTuito][ES] n=2397 | clases={0: 1598, 1: 799} | manifest=OK
[DATA][RoBERTuito][CU] n=2400 | clases={0: 1600, 1: 800} | manifest=OK


## 4. Tokenizador y datos (igual que v4)

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print("Tokenizer:", tokenizer.__class__.__name__)
print("do_lower_case:", getattr(tokenizer, "do_lower_case", "gestionado por tokenizer"))
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer, pad_to_multiple_of=8 if torch.cuda.is_available() else None
)


class DatasetTokenizado(Dataset):
    def __init__(self, textos, etiquetas):
        self.encoded = tokenizer(
            [str(t) for t in textos], truncation=True, max_length=MAX_LENGTH, padding=False
        )
        self.labels = np.asarray(etiquetas, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encoded.items()}
        item["labels"] = int(self.labels[i])
        return item


def cache_dataset(df):
    y = df[LABEL_COL].to_numpy(dtype=np.int64)
    return {prep: DatasetTokenizado(df[col].tolist(), y) for prep, col in TEXT_COLS.items()}, y

Tokenizer: TokenizersBackend
do_lower_case: gestionado por tokenizer


### 4.1 `GROUP_ID` (igual que v4: mismos grupos, mismos folds)

In [5]:
from collections import defaultdict


def colapsar_espacios(texto):
    return re.sub(r"\s+", " ", str(texto)).strip()


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def clave_tokens(ids):
    """Representación directa de los input_ids, sin hashes."""
    return ",".join(map(str, ids))


def construir_grupos_robertuito(df, variante, tokenizador=tokenizer, max_length=MAX_LENGTH):
    """Agrupa entradas equivalentes para evitar que crucen train/validación.

    Agrupación propia del modelo: texto original, cada representación de
    entrada (TEXT_COLS) y su secuencia tokenizada (truncada a MAX_LENGTH).
    """
    trabajo = df.copy().reset_index(drop=True)
    trabajo["KEY_ORIGINAL"] = trabajo[ORIGINAL_COL].map(colapsar_espacios)
    for prep, columna in TEXT_COLS.items():
        trabajo[f"KEY_{prep}"] = trabajo[columna].map(colapsar_espacios)

    for prep, columna in TEXT_COLS.items():
        ids = tokenizador(
            trabajo[columna].astype(str).tolist(),
            truncation=True,
            max_length=max_length,
            padding=False,
        )["input_ids"]
        trabajo[f"KEY_TOK_{prep}"] = [clave_tokens(x) for x in ids]

    representaciones = ["ORIGINAL", *TEXT_COLS, *[f"TOK_{p}" for p in TEXT_COLS]]
    key_cols = [f"KEY_{r}" for r in representaciones]
    uf = UnionFind(len(trabajo))

    for col in key_cols:
        first_seen = {}
        for idx, key in enumerate(trabajo[col].tolist()):
            if not key:
                continue
            if key in first_seen:
                uf.union(idx, first_seen[key])
            else:
                first_seen[key] = idx

    componentes = defaultdict(list)
    for idx in range(len(trabajo)):
        componentes[uf.find(idx)].append(idx)

    group_ids = [None] * len(trabajo)

    componentes_ordenados = sorted(componentes.values(), key=lambda miembros: min(miembros))

    for numero_grupo, miembros in enumerate(componentes_ordenados, start=1):
        gid = f"{variante}_G{numero_grupo:04d}"
        for idx in miembros:
            group_ids[idx] = gid
    trabajo[GROUP_COL] = group_ids

    resumen = []
    for nombre, col in zip(representaciones, key_cols):
        sizes = trabajo.groupby(col).size()
        repetidos = sizes[sizes > 1]
        resumen.append(
            {
                "variante": variante,
                "representacion": nombre,
                "filas_en_repeticiones": int(trabajo[col].isin(repetidos.index).sum()),
                "grupos_repetidos": int(len(repetidos)),
            }
        )

    gstats = trabajo.groupby(GROUP_COL)[LABEL_COL].agg(["size", "nunique"])
    conflictivos = set(gstats[(gstats["size"] > 1) & (gstats["nunique"] > 1)].index)
    detalle_conflictos = trabajo[trabajo[GROUP_COL].isin(conflictivos)].copy()

    resumen = pd.DataFrame(resumen)
    resumen["n_train"] = len(trabajo)
    resumen["n_group_id"] = trabajo[GROUP_COL].nunique()
    resumen["max_group_size"] = int(trabajo.groupby(GROUP_COL).size().max())
    resumen["grupos_con_labels_mixtos"] = len(conflictivos)
    return trabajo, resumen, detalle_conflictos


TRAIN_AGRUPADO = {}
AUDITORIA_GRUPOS = []
DETALLES_CONFLICTOS = []

# Se construyen los grupos de las tres variantes antes de una corrida larga.
for variante in ["mx", "es", "cu"]:
    df_audit = cargar_train(variante)
    agrupado, resumen, conflictos = construir_grupos_robertuito(df_audit, variante)
    TRAIN_AGRUPADO[variante] = agrupado
    AUDITORIA_GRUPOS.append(resumen)
    if not conflictos.empty:
        conflictos["variante"] = variante
        DETALLES_CONFLICTOS.append(conflictos)

df_auditoria_grupos = pd.concat(AUDITORIA_GRUPOS, ignore_index=True)
df_auditoria_grupos.to_csv(RUN_DIR / "auditoria_grupos_train.csv", index=False)
display(df_auditoria_grupos)

if DETALLES_CONFLICTOS:
    df_conflictos_grupo = pd.concat(DETALLES_CONFLICTOS, ignore_index=True)
    df_conflictos_grupo.to_csv(RUN_DIR / "auditoria_grupos_labels_mixtos.csv", index=False)

for variante in ["mx", "es", "cu"]:
    df_g = TRAIN_AGRUPADO[variante]
    n_mixtos = int(df_g.groupby(GROUP_COL)[LABEL_COL].nunique().gt(1).sum())
    print(
        f"[AUDIT][RoBERTuito][{variante.upper()}] n={len(df_g)} | "
        f"groups={df_g[GROUP_COL].nunique()} | "
        f"max_group={df_g.groupby(GROUP_COL).size().max()} | labels_mixtos={n_mixtos}"
    )

    if n_mixtos > 0:
        print(
            f"  ADVERTENCIA: {n_mixtos} GROUP_ID contienen etiquetas mixtas. "
            "No es data leakage porque el grupo permanece íntegro dentro de un fold, "
            "pero representa ambigüedad/ruido de etiqueta y debe documentarse."
        )

,variante,representacion,filas_en_repeticiones,grupos_repetidos,n_train,n_group_id,max_group_size,grupos_con_labels_mixtos
0,mx,ORIGINAL,0,0,2399,2397,3,0
1,mx,PYSENTIMIENTO,0,0,2399,2397,3,0
2,mx,TOK_PYSENTIMIENTO,3,1,2399,2397,3,0
3,es,ORIGINAL,0,0,2397,2306,17,1
4,es,PYSENTIMIENTO,131,41,2397,2306,17,1
5,es,TOK_PYSENTIMIENTO,127,41,2397,2306,17,1
6,cu,ORIGINAL,6,3,2400,2397,2,0
7,cu,PYSENTIMIENTO,6,3,2400,2397,2,0
8,cu,TOK_PYSENTIMIENTO,6,3,2400,2397,2,0


[AUDIT][RoBERTuito][MX] n=2399 | groups=2397 | max_group=3 | labels_mixtos=0
[AUDIT][RoBERTuito][ES] n=2397 | groups=2306 | max_group=17 | labels_mixtos=1
  ADVERTENCIA: 1 GROUP_ID contienen etiquetas mixtas. No es data leakage porque el grupo permanece íntegro dentro de un fold, pero representa ambigüedad/ruido de etiqueta y debe documentarse.
[AUDIT][RoBERTuito][CU] n=2400 | groups=2397 | max_group=2 | labels_mixtos=0


### 4.2 Pre-flight (igual que v4; exige GPU)

In [6]:
def comprobar_preflight_robertuito():
    print("=" * 78)
    print("PRE-FLIGHT — ROBERTUITO GROUPED NESTED CV")
    print("=" * 78)

    if MODO == "completo":
        esperado = (N_OUTER, N_INNER, N_TRIALS, N_STARTUP_TRIALS)

        if esperado != (5, 3, 20, 5):
            raise ValueError(
                f"Configuración inesperada: {esperado}; " "se esperaba 5×3, 20 trials, startup=5."
            )

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA no disponible. Verifica entorno/PyTorch antes de entrenar RoBERTuito."
        )

    for variante in ["mx", "es", "cu"]:
        _, manifest_row = validar_archivo_train_con_manifest(variante)

        df = TRAIN_AGRUPADO[variante]

        if len(df) != int(manifest_row["train_final"]):
            raise ValueError(f"{variante}: train agrupado no coincide " "con el manifest.")

        if df[ID_COL].duplicated().any():
            raise ValueError(f"{variante}: IDs duplicados.")

        if df[GROUP_COL].isna().any():
            raise ValueError(f"{variante}: GROUP_ID nulo.")

        if not set(df[LABEL_COL].unique()).issubset({0, 1}):
            raise ValueError(f"{variante}: etiquetas fuera de 0/1.")

        n_mixtos = int(df.groupby(GROUP_COL)[LABEL_COL].nunique().gt(1).sum())

        if n_mixtos:
            print(
                f"[AUDIT][RoBERTuito][{variante.upper()}] "
                f"ADVERTENCIA: {n_mixtos} GROUP_ID "
                "con etiquetas mixtas; se documentan "
                "como ruido/ambigüedad."
            )

    print("Preprocessing:", PIPELINE_VERSION_ESPERADA)
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Nested: {N_OUTER}×{N_INNER} | Optuna={N_TRIALS} | startup={N_STARTUP_TRIALS}")
    print("Train final:", {v: len(TRAIN_AGRUPADO[v]) for v in ["mx", "es", "cu"]})
    print("PRE-FLIGHT DE DATOS: OK")


comprobar_preflight_robertuito()

PRE-FLIGHT — ROBERTUITO GROUPED NESTED CV
[AUDIT][RoBERTuito][ES] ADVERTENCIA: 1 GROUP_ID con etiquetas mixtas; se documentan como ruido/ambigüedad.
Preprocessing: transformers_comun_v2
GPU: NVIDIA RTX 6000 Ada Generation
Nested: 5×3 | Optuna=20 | startup=5
Train final: {'mx': 2399, 'es': 2397, 'cu': 2400}
PRE-FLIGHT DE DATOS: OK


## 5. Métricas, modelo y trainer (igual que v4)

In [7]:
def establecer_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def liberar_memoria():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def metricas(prediction):
    logits, labels = prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    y_pred = np.asarray(logits).argmax(axis=-1)
    return {
        "f1_macro": f1_score(labels, y_pred, average="macro"),
        "accuracy": accuracy_score(labels, y_pred),
        "precision_macro": precision_score(labels, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, y_pred, average="macro", zero_division=0),
    }


def pesos_balanceados(y):
    classes = np.array([0, 1])
    w = compute_class_weight("balanced", classes=classes, y=np.asarray(y))
    return torch.tensor(w, dtype=torch.float32)


def crear_modelo(seed, classifier_dropout=0.2):
    establecer_semilla(seed)
    with (RUN_DIR / "carga_modelos.log").open("a", encoding="utf-8") as log, redirect_stdout(
        log
    ), redirect_stderr(log):
        config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)
        if config.model_type != "roberta":
            raise ValueError(f"Se esperaba arquitectura RoBERTa, recibida: {config.model_type}")
        # RoBERTa: dropout de la cabeza de clasificación.
        config.classifier_dropout = float(classifier_dropout)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
    return model


class ClasificadorTrainer(Trainer):
    """v4: pérdida ponderada opcional + label smoothing (solo en train) + LR decay por capas."""

    def __init__(self, *args, class_weights=None, label_smoothing=0.0, lr_decay=1.0, **kwargs):
        self.class_weights = class_weights
        self.label_smoothing = float(label_smoothing)
        self.lr_decay = float(lr_decay)
        super().__init__(*args, **kwargs)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs["labels"]
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        weights = (
            self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
        )
        # Smoothing SOLO al entrenar: la pérdida de validación sigue siendo entropía cruzada estándar.
        ls = self.label_smoothing if model.training else 0.0
        loss = F.cross_entropy(outputs.logits, labels, weight=weights, label_smoothing=ls)
        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self, *args, **kwargs):
        if self.optimizer is not None:
            return self.optimizer
        if self.lr_decay >= 1.0:
            return super().create_optimizer(*args, **kwargs)
        # Capas: BERT/RoBERTa `encoder.layer.N`, DistilBERT `transformer.layer.N`.
        patron = re.compile(r"\.layer\.(\d+)\.")
        n_capas = 1 + max(
            int(m.group(1)) for n, _ in self.model.named_parameters() if (m := patron.search(n))
        )
        base_lr, wd = self.args.learning_rate, self.args.weight_decay
        grupos = {}
        for nombre, p in self.model.named_parameters():
            if not p.requires_grad:
                continue
            m = patron.search(nombre)
            if m:
                profundidad = n_capas - int(m.group(1))  # última capa = 1
            elif "embeddings" in nombre:
                profundidad = n_capas + 1  # embeddings = LR más bajo
            else:
                profundidad = 0  # cabeza de clasificación / pooler = LR base
            sin_wd = nombre.endswith("bias") or "LayerNorm" in nombre or "layer_norm" in nombre
            clave = (base_lr * self.lr_decay**profundidad, 0.0 if sin_wd else wd)
            grupos.setdefault(clave, []).append(p)
        self.optimizer = torch.optim.AdamW(
            [{"params": ps, "lr": lr, "weight_decay": w} for (lr, w), ps in grupos.items()]
        )
        return self.optimizer


class ParadaTemprana(TrainerCallback):
    """Early stopping por F1-Macro de validación sin guardar checkpoints (Inner Folds)."""

    def __init__(self, paciencia=2):
        self.paciencia, self.mejor, self.sin_mejora = int(paciencia), -np.inf, 0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        f1 = (metrics or {}).get("eval_f1_macro")
        if f1 is None:
            return control
        if f1 > self.mejor:
            self.mejor, self.sin_mejora = f1, 0
        else:
            self.sin_mejora += 1
            if self.sin_mejora >= self.paciencia:
                control.should_training_stop = True
        return control


class DetenerEnEpoca(TrainerCallback):
    """Refit/test con el MISMO scheduler que la búsqueda: se detiene en la época elegida."""

    def __init__(self, epocas):
        self.epocas = int(epocas)

    def on_epoch_end(self, args, state, control, **kwargs):
        if state.epoch is not None and state.epoch >= self.epocas - 1e-6:
            control.should_training_stop = True
        return control


## 6. Argumentos de entrenamiento (igual que v4)

In [8]:
def argumentos_entrenamiento(carpeta, cfg, seed, con_validacion=True, epochs=None):
    params = {
        "output_dir": str(carpeta),
        "learning_rate": cfg["learning_rate"],
        "per_device_train_batch_size": cfg["batch_size"],
        "per_device_eval_batch_size": cfg["batch_size"],
        "num_train_epochs": epochs if epochs is not None else cfg["num_train_epochs"],
        "weight_decay": cfg["weight_decay"],
        "warmup_ratio": 0.10,
        "optim": "adamw_torch",
        # Trainer recoge loss por época en state.log_history;
        # PrinterCallback se elimina para no saturar VS Code.
        "logging_strategy": "epoch",
        "save_strategy": "epoch",  # Inner y refit: reinicio desde última época completa.
        "load_best_model_at_end": bool(con_validacion),
        "save_total_limit": 2,  # último + mejor (si son diferentes)
        "save_only_model": False,  # Estado optimizer/scheduler requerido para reanudar.
        "save_safetensors": True,
        "report_to": "none",
        "disable_tqdm": True,
        # Necesario para que EarlyStoppingCallback conserve su contador
        # al reanudar desde un checkpoint.
        "restore_callback_states_from_checkpoint": True,
        "dataloader_num_workers": 0,
        "seed": seed,
        "data_seed": seed,
        "bf16": USE_BF16,
        "fp16": USE_FP16,
    }
    if con_validacion:
        # v4 — Inner Folds sin checkpoints: el modelo interno se descarta y F1/mejor época
        # se leen de la curva. El early stopping lo hace ParadaTemprana.
        params.update({"save_strategy": "no", "load_best_model_at_end": False})
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in sig:
        params["eval_strategy"] = "epoch" if con_validacion else "no"
    elif "evaluation_strategy" in sig:
        params["evaluation_strategy"] = "epoch" if con_validacion else "no"
    else:
        raise RuntimeError("Esta versión de Transformers no admite evaluación por época.")
    return TrainingArguments(**{k: v for k, v in params.items() if k in sig})


def mejor_epoca(trainer, max_epochs):
    evaluaciones = [x for x in trainer.state.log_history if "eval_f1_macro" in x and "epoch" in x]
    if not evaluaciones:
        raise RuntimeError("No se encontró F1 de validación por época.")
    mejor = max(evaluaciones, key=lambda x: x["eval_f1_macro"])
    return int(np.clip(round(float(mejor["epoch"])), 1, max_epochs))


def predecir(trainer, dataset):
    logits = trainer.predict(dataset).predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    return np.argmax(logits, axis=-1)

## 7. Utilidades (copiadas de la sección 7 de la v4)

In [9]:
def entrenador_silencioso(trainer):
    trainer.remove_callback(PrinterCallback)
    trainer.remove_callback(ProgressCallback)
    return trainer


def csv_atomico(df, ruta):
    ruta = Path(ruta)
    tmp = ruta.with_name(ruta.name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, ruta)


## 8. Pilotos v5

Cada fold imprime una línea `[CHECK]` con las capas congeladas y el % de parámetros entrenables. **Revísala en el primer fold de P1:** debe decir `capas congeladas=[0, 1, 2]`. La celda muestra el tiempo por fold y una estimación del tiempo restante.


In [ ]:
# ============================================================
# PILOTOS v5 — congelamiento de capas y label smoothing
# Solo train oficial: mismos 3 Inner Folds, misma configuración final y
# mismas semillas que el diagnóstico de convergencia v4 (sección 16-bis).
# Sin early stopping, 4 evaluaciones por época, train y validación en modo eval.
# NO toca el test. NO selecciona hiperparámetros: solo descarta opciones dañinas.
#
#   P0 = configuración final v4 tal cual (línea base, misma máquina)
#   P1 = P0 + congelar embeddings y el ~25 % inferior de capas
#   P2 = P0 + congelar embeddings y el 50 % inferior de capas
#   P3 = P0 con label smoothing = 0 (sin congelar)
#
# Respaldo: Ingle et al. (2022) — congelar 25–50 %; Merchant et al. (2020) —
# congelar embeddings + k capas inferiores; Desai & Durrett (2020) — label smoothing.
# Se puede interrumpir y volver a ejecutar: reutiliza los folds ya terminados.
# ============================================================
import math
import time

VARIANTES_PILOTO = ["mx", "cu"]
PILOTOS_A_CORRER = ["P0", "P1", "P2", "P3"]  # quitar "P0" = reutilizar el diagnóstico v4
EVALS_POR_EPOCA = 4
REHACER = False          # True = borra los CSV previos de los pilotos y vuelve a entrenar
UMBRAL_F1 = 0.01         # criterio fijado ANTES de ver resultados

_cfg_hf = AutoConfig.from_pretrained(MODEL_NAME)
N_CAPAS = int(getattr(_cfg_hf, "num_hidden_layers", 0) or getattr(_cfg_hf, "n_layers"))
CAPAS_25 = max(1, round(N_CAPAS * 0.25))   # BETO/RoBERTuito (12) -> 3 | DistilBETO (6) -> 2
CAPAS_50 = N_CAPAS // 2                    # BETO/RoBERTuito (12) -> 6 | DistilBETO (6) -> 3

PILOTOS = {
    "P0": {"nombre": "v4 (sin congelar, LS 0.1)", "congelar_capas": 0},
    "P1": {"nombre": f"congelar emb + {CAPAS_25}/{N_CAPAS} capas", "congelar_capas": CAPAS_25},
    "P2": {"nombre": f"congelar emb + {CAPAS_50}/{N_CAPAS} capas", "congelar_capas": CAPAS_50},
    "P3": {"nombre": "label smoothing 0", "congelar_capas": 0, "label_smoothing": 0.0},
}
COLORES = {"P0": "black", "P1": "C0", "P2": "C2", "P3": "C3"}

BASE = NOMBRE_EXPERIMENTO.lower()
PILOTOS_DIR = RESULTS_DIR / "pilotos_v5"
DIAG_V4_DIR = RESULTS_DIR / f"diag_{BASE}_diagnostico"
PATRON_CAPA = re.compile(r"\.layer\.(\d+)\.")

print(f"[PILOTOS][{NOMBRE_EXPERIMENTO}] capas={N_CAPAS} | P1 congela {CAPAS_25} | "
      f"P2 congela {CAPAS_50} | variantes={VARIANTES_PILOTO} | pilotos={PILOTOS_A_CORRER}")


def congelar_capas(model, n_capas):
    """Congela embeddings + capas Transformer 0..n_capas-1 (Merchant et al., 2020)."""
    if n_capas <= 0:
        return 0, []
    congelados, capas = 0, set()
    for nombre, p in model.named_parameters():
        m = PATRON_CAPA.search(nombre)
        if "embeddings" in nombre or (m and int(m.group(1)) < n_capas):
            p.requires_grad = False
            congelados += p.numel()
            if m:
                capas.add(int(m.group(1)))
    return congelados, sorted(capas)


def ece_binaria(probs, labels, n_bins=10):
    """Expected Calibration Error con 10 bins de igual ancho sobre la confianza."""
    confianza = probs.max(axis=1)
    acierto = (probs.argmax(axis=1) == labels).astype(float)
    bordes = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bordes[:-1], bordes[1:]):
        mask = (confianza > lo) & (confianza <= hi)
        if mask.any():
            ece += mask.mean() * abs(acierto[mask].mean() - confianza[mask].mean())
    return float(ece)


def metricas_piloto(prediction):
    salida = metricas(prediction)
    logits, labels = prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    probs = torch.softmax(torch.tensor(np.asarray(logits), dtype=torch.float32), dim=-1).numpy()
    salida["ece"] = ece_binaria(probs, np.asarray(labels))
    return salida


def _args_piloto(carpeta, cfg, seed, eval_steps, epocas):
    params = dict(
        output_dir=str(carpeta),
        learning_rate=cfg["learning_rate"],
        per_device_train_batch_size=cfg["batch_size"],
        per_device_eval_batch_size=64,
        num_train_epochs=epocas,
        weight_decay=cfg["weight_decay"],
        warmup_ratio=0.10,
        optim="adamw_torch",
        logging_strategy="steps",
        logging_steps=eval_steps,
        eval_steps=eval_steps,
        save_strategy="no",
        load_best_model_at_end=False,
        report_to="none",
        disable_tqdm=True,
        dataloader_num_workers=0,
        seed=seed,
        data_seed=seed,
        bf16=USE_BF16,
        fp16=USE_FP16,
    )
    sig = inspect.signature(TrainingArguments.__init__).parameters
    params["eval_strategy" if "eval_strategy" in sig else "evaluation_strategy"] = "steps"
    return TrainingArguments(**{k: v for k, v in params.items() if k in sig})


def _historia_a_tabla_piloto(log_history, metricas_paso0, pasos_epoca):
    mapa = {
        "eval_train_loss": "train_loss",
        "eval_val_loss": "val_loss",
        "eval_train_f1_macro": "train_f1",
        "eval_val_f1_macro": "val_f1",
        "eval_train_ece": "train_ece",
        "eval_val_ece": "val_ece",
        "loss": "train_loss_running",
    }
    filas = {0: {"step": 0, **metricas_paso0}}
    for e in log_history:
        if "step" not in e:
            continue
        fila = filas.setdefault(e["step"], {"step": e["step"]})
        for origen, destino in mapa.items():
            if origen in e:
                fila[destino] = float(e[origen])
    tabla = pd.DataFrame(list(filas.values())).sort_values("step")
    tabla = tabla[tabla["val_loss"].notna()].reset_index(drop=True)
    tabla["epoch"] = tabla["step"] / pasos_epoca
    tabla["pasos_epoca"] = pasos_epoca
    return tabla


def entrenar_fold_piloto(clave, variante, cfg, ds, y, tr, val, numero):
    seed_fold = SEED + SEED * 100 + numero  # misma semilla que el diagnóstico v4
    rng = np.random.default_rng(seed_fold)
    tr_eval = np.sort(rng.choice(tr, size=min(len(val), len(tr)), replace=False))
    pasos_epoca = math.ceil(len(tr) / cfg["batch_size"])
    eval_steps = max(1, pasos_epoca // EVALS_POR_EPOCA)
    epocas = int(cfg["num_train_epochs"])
    carpeta = TMP_DIR / f"pil_{clave}_{variante}_in{numero}"

    model = crear_modelo(seed_fold, cfg["classifier_dropout"])
    n_cong, capas_cong = congelar_capas(model, PILOTOS[clave]["congelar_capas"])
    n_total = sum(p.numel() for p in model.parameters())
    n_entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)
    cw = pesos_balanceados(y[tr]) if cfg["loss_mode"] == "balanced" else None
    ds_train_eval = Subset(ds, tr_eval.tolist())
    ds_val = Subset(ds, val.tolist())
    trainer = None
    try:
        trainer = entrenador_silencioso(
            ClasificadorTrainer(
                model=model,
                args=_args_piloto(carpeta, cfg, seed_fold, eval_steps, epocas),
                train_dataset=Subset(ds, tr.tolist()),
                eval_dataset={"train": ds_train_eval, "val": ds_val},
                data_collator=data_collator,
                compute_metrics=metricas_piloto,
                class_weights=cw,
                label_smoothing=cfg.get("label_smoothing", 0.0),
                lr_decay=cfg.get("lr_decay", 1.0),
            )
        )
        print(
            f"[CHECK] {clave} {variante.upper()} inner {numero} | "
            f"capas congeladas={capas_cong if capas_cong else 'ninguna'} | "
            f"entrenables={n_entrenables / n_total:.1%} | "
            f"LS={trainer.label_smoothing} | LRD={trainer.lr_decay} | épocas={epocas}"
        )
        m_tr = trainer.evaluate(eval_dataset=ds_train_eval, metric_key_prefix="eval_train")
        m_va = trainer.evaluate(eval_dataset=ds_val, metric_key_prefix="eval_val")
        paso0 = {
            "train_loss": m_tr["eval_train_loss"],
            "train_f1": m_tr["eval_train_f1_macro"],
            "train_ece": m_tr["eval_train_ece"],
            "val_loss": m_va["eval_val_loss"],
            "val_f1": m_va["eval_val_f1_macro"],
            "val_ece": m_va["eval_val_ece"],
        }
        trainer.state.log_history = []
        trainer.train()
        tabla = _historia_a_tabla_piloto(trainer.state.log_history, paso0, pasos_epoca)
    finally:
        if trainer is not None:
            del trainer
        del model
        liberar_memoria()
        shutil.rmtree(carpeta, ignore_errors=True)
    return tabla


def ejecutar_pilotos():
    configs = json.loads((RESULTS_DIR / f"{BASE}_config_final.json").read_text(encoding="utf-8"))
    if REHACER and PILOTOS_DIR.exists():
        shutil.rmtree(PILOTOS_DIR)
    PILOTOS_DIR.mkdir(parents=True, exist_ok=True)

    tareas = [(v, c, n) for v in VARIANTES_PILOTO for c in PILOTOS_A_CORRER for n in (1, 2, 3)]
    pendientes = [t for t in tareas if not (PILOTOS_DIR / t[1] / f"{t[0]}_inner{t[2]}.csv").exists()]
    print(f"[PILOTOS] fine-tunings totales={len(tareas)} | pendientes={len(pendientes)}")

    partes, duraciones, hechos = [], [], 0
    for variante in VARIANTES_PILOTO:
        df = TRAIN_AGRUPADO[variante].copy()
        cache, y = cache_dataset(df)
        groups = df[GROUP_COL].astype(str).to_numpy()
        # Mismos splits que la optimización final y el diagnóstico v4 (k_inner=3, seed=SEED).
        skf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
        splits = list(skf.split(np.zeros(len(y)), y, groups=groups))

        claves = list(PILOTOS_A_CORRER)
        if "P0" not in claves:
            for numero in (1, 2, 3):
                t = pd.read_csv(DIAG_V4_DIR / f"{variante}_inner{numero}.csv")
                t["piloto"] = "P0"
                partes.append(t)
            print(f"[PILOTOS] {variante.upper()} P0: reutilizado del diagnóstico v4 (sin ECE)")

        for clave in claves:
            cfg = {k: v for k, v in configs[variante].items() if k != "refit_epochs"}
            cfg.update({k: v for k, v in PILOTOS[clave].items() if k in cfg})
            ds = cache[cfg["preprocessing"]]
            for numero, (tr, val) in enumerate(splits, start=1):
                ruta_csv = PILOTOS_DIR / clave / f"{variante}_inner{numero}.csv"
                ruta_csv.parent.mkdir(parents=True, exist_ok=True)
                if ruta_csv.exists():
                    partes.append(pd.read_csv(ruta_csv))
                    print(f"[PILOTOS] {clave} {variante.upper()} inner {numero}: reutilizado")
                    continue
                t0 = time.time()
                tabla = entrenar_fold_piloto(clave, variante, cfg, ds, y, tr, val, numero)
                tabla.insert(0, "inner_fold", numero)
                tabla.insert(0, "variante", variante)
                tabla.insert(0, "piloto", clave)
                csv_atomico(tabla, ruta_csv)
                partes.append(tabla)
                duraciones.append(time.time() - t0)
                hechos += 1
                restantes = len(pendientes) - hechos
                eta = np.mean(duraciones) * restantes / 60
                print(f"[PILOTOS] {clave} {variante.upper()} inner {numero}: "
                      f"{duraciones[-1] / 60:.1f} min | faltan {restantes} (~{eta:.0f} min)")
                if AVISO_SONORO and os.name == "nt" and restantes == 0:
                    import winsound
                    winsound.Beep(880, 600)

    pil = pd.concat(partes, ignore_index=True)
    pil["epoch_r"] = (pil["epoch"] * EVALS_POR_EPOCA).round() / EVALS_POR_EPOCA
    csv_atomico(pil, RESULTS_DIR / f"{BASE}_pilotos_convergencia.csv")
    return pil


def resumir_pilotos(pil):
    filas = []
    for (variante, clave), d in pil.groupby(["variante", "piloto"]):
        media = d.groupby("epoch_r", as_index=False).mean(numeric_only=True)
        i_loss, i_f1 = media["val_loss"].idxmin(), media["val_f1"].idxmax()
        filas.append({
            "modelo": NOMBRE_EXPERIMENTO,
            "variante": variante,
            "piloto": clave,
            "descripcion": PILOTOS[clave]["nombre"],
            "max_val_f1": media.loc[i_f1, "val_f1"],
            "epoca_max_f1": media.loc[i_f1, "epoch_r"],
            "train_f1_en_max": media.loc[i_f1, "train_f1"],
            "brecha_f1": media.loc[i_f1, "train_f1"] - media.loc[i_f1, "val_f1"],
            "min_val_loss": media.loc[i_loss, "val_loss"],
            "epoca_min_loss": media.loc[i_loss, "epoch_r"],
            "val_loss_final": media["val_loss"].iloc[-1],
            "subida_val_loss": media["val_loss"].iloc[-1] - media.loc[i_loss, "val_loss"],
            "val_ece_en_max": media.loc[i_f1, "val_ece"] if "val_ece" in media else np.nan,
        })
    res = pd.DataFrame(filas)
    base0 = res[res["piloto"] == "P0"].set_index("variante")
    res["delta_f1_vs_P0"] = res.apply(lambda r: r["max_val_f1"] - base0.loc[r["variante"], "max_val_f1"], axis=1)
    res["delta_brecha_vs_P0"] = res.apply(lambda r: r["brecha_f1"] - base0.loc[r["variante"], "brecha_f1"], axis=1)
    res["no_daña_f1"] = res["delta_f1_vs_P0"] >= -UMBRAL_F1
    csv_atomico(res, RESULTS_DIR / f"{BASE}_pilotos_resumen.csv")
    return res


def figura_pilotos(pil):
    variantes = [v for v in ["mx", "es", "cu"] if v in set(pil["variante"])]
    fig, axes = plt.subplots(2, len(variantes), figsize=(5.2 * len(variantes), 7.8),
                             sharex="col", squeeze=False)
    for j, variante in enumerate(variantes):
        for clave in [c for c in PILOTOS if c in set(pil["piloto"])]:
            d = pil[(pil["variante"] == variante) & (pil["piloto"] == clave)]
            media = d.groupby("epoch_r", as_index=False).mean(numeric_only=True)
            color = COLORES[clave]
            etiqueta = f"{clave}: {PILOTOS[clave]['nombre']}"
            for fila, met in [(0, "loss"), (1, "f1")]:
                ax = axes[fila, j]
                ax.plot(media["epoch_r"], media[f"val_{met}"], color=color, lw=2.0,
                        label=f"{etiqueta} (val.)")
                ax.plot(media["epoch_r"], media[f"train_{met}"], color=color, lw=1.0, ls="--",
                        alpha=0.7, label=f"{etiqueta} (train)")
        axes[0, j].set_title(variante.upper(), fontsize=11)
        axes[1, j].set_xlabel("Época")
        for ax in axes[:, j]:
            ax.grid(alpha=0.25)
    axes[0, 0].set_ylabel("Pérdida (entropía cruzada, modo eval)")
    axes[1, 0].set_ylabel("F1-Macro")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=2, fontsize=8, frameon=False)
    fig.suptitle(
        f"{NOMBRE_EXPERIMENTO} — pilotos de regularización (media de 3 Inner Folds; "
        f"línea continua = validación, discontinua = train)",
        fontsize=10.5,
    )
    fig.tight_layout(rect=(0, 0.1, 1, 1))
    for ext in ("png", "pdf"):
        fig.savefig(RESULTS_DIR / f"{BASE}_pilotos_curvas.{ext}", dpi=200, bbox_inches="tight")
    display(fig)
    plt.close(fig)


if not (RESULTS_DIR / f"{BASE}_config_final.json").exists():
    raise FileNotFoundError(f"Falta {BASE}_config_final.json en {RESULTS_DIR}.")
PILOTOS_RESULTADOS = ejecutar_pilotos()
PILOTOS_RESUMEN = resumir_pilotos(PILOTOS_RESULTADOS)
display(PILOTOS_RESUMEN.round(4))
figura_pilotos(PILOTOS_RESULTADOS)
print(f"✅ [{NOMBRE_EXPERIMENTO}] Pilotos terminados — {datetime.now():%Y-%m-%d %H:%M:%S}")
print(f"Resultados en: {PILOTOS_DIR.resolve()}")


[PILOTOS][RoBERTuito] capas=12 | P1 congela 3 | P2 congela 6 | variantes=['mx', 'cu'] | pilotos=['P0', 'P1', 'P2', 'P3']
[PILOTOS] fine-tunings totales=24 | pendientes=24
[CHECK] P0 MX inner 1 | capas congeladas=ninguna | entrenables=100.0% | LS=0.1 | LRD=0.9 | épocas=6
[PILOTOS] P0 MX inner 1: 0.7 min | faltan 23 (~15 min)
[CHECK] P0 MX inner 2 | capas congeladas=ninguna | entrenables=100.0% | LS=0.1 | LRD=0.9 | épocas=6
[PILOTOS] P0 MX inner 2: 0.7 min | faltan 22 (~15 min)
[CHECK] P0 MX inner 3 | capas congeladas=ninguna | entrenables=100.0% | LS=0.1 | LRD=0.9 | épocas=6
[PILOTOS] P0 MX inner 3: 0.7 min | faltan 21 (~14 min)
[CHECK] P1 MX inner 1 | capas congeladas=[0, 1, 2] | entrenables=59.2% | LS=0.1 | LRD=0.9 | épocas=6
[PILOTOS] P1 MX inner 1: 0.6 min | faltan 20 (~13 min)
[CHECK] P1 MX inner 2 | capas congeladas=[0, 1, 2] | entrenables=59.2% | LS=0.1 | LRD=0.9 | épocas=6
[PILOTOS] P1 MX inner 2: 0.6 min | faltan 19 (~12 min)
[CHECK] P1 MX inner 3 | capas congeladas=[0, 1, 2] |

## 9. Qué mandar de vuelta

Copia de vuelta a la Mac la carpeta `data/resultados_<modelo>_grouped_5x3_tpe20_v4/` actualizada. Basta con:
- `<modelo>_pilotos_resumen.csv`
- `<modelo>_pilotos_convergencia.csv`
- `<modelo>_pilotos_curvas.png`
